In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm

from retriever import Retriever
from prompt_builder import PromptBuilder
from gemini_integration import load_api_keys, init_gemini, call_gemini
from utils import DEFAULT_EMBEDDING_MODEL, DEFAULT_INDEX_NAME, DEFAULT_API_KEYS_PATH

TESTSET_CSV = "./evaluation/haifa_testset.csv"
OUT_CSV = "./evaluation/haifa_rag_strategies_raw_answers.csv"

TOP_K_RETRIEVAL = 8   # before reranking
TOP_K_FINAL = 5       # how many chunks to include in the final prompt

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

# load Gemini
api_keys = load_api_keys("api_keys.json")
gemini_model = init_gemini(api_keys, model_name="gemini-2.5-flash")

# load Retriever
retriever = Retriever(
    api_keys_path=DEFAULT_API_KEYS_PATH,
    embedding_model_name=DEFAULT_EMBEDDING_MODEL,
    index_name=DEFAULT_INDEX_NAME,
    namespace=None,  # if there is a specific namespace for Haifa, put it here
)

# Prompt builder "regular" for RAG use
prompt_builder = PromptBuilder()


In [ ]:
# Query Rephrase & Enrich

def rephrase_query(question: str) -> str:
    prompt = f"""
אתה עוזר AI. קבל שאלה של תושב חיפה וכתוב אותה מחדש בצורה ברורה ומדויקת יותר,
ללא שינוי המשמעות.

שאלה מקורית:
\"\"\"{question}\"\"\"

החזר רק את הניסוח המשופר, ללא הסברים נוספים.
"""
    resp = call_gemini(gemini_model, prompt)
    return resp.strip() or question


def enrich_query(question: str) -> str:
    prompt = f"""
קבל שאלה של תושב חיפה על שירות עירוני. על בסיס השאלה,
הצע ניסוח מורחב שמוסיף מילות מפתח ורמזים חשובים לחיפוש במסמכים.

שאלה:
\"\"\"{question}\"\"\"

הנחיות:
1. אל תשנה את המשמעות.
2. הוסף מונחים נלווים (למשל: "ארנונה, תשלומים, חיוב, city4u").
3. החזר שורה אחת בלבד, עם השאלה המורחבת.

החזר רק את השאלה המורחבת, ללא הסברים.
"""
    resp = call_gemini(gemini_model, prompt)
    return resp.strip() or question


In [ ]:
# Rerank chunks with Gemini

def rerank_chunks(question: str, chunks: list, top_k: int = TOP_K_FINAL) -> list:
    """
    takes a question and a list of chunks as returned by the retriever,
    returns a sublist of the most relevant chunks according to Gemini.
    """
    if not chunks:
        return []

    # prepare a summary of each chunk for the prompt
    lines = []
    for i, ch in enumerate(chunks, 1):
        text = ch.get("chunk_text_only") or ch.get("text", "")
        text = str(text).replace("\n", " ")
        if len(text) > 350:
            text = text[:350] + "..."
        title = ch.get("title", "")
        url = ch.get("url", "")
        meta = []
        if title:
            meta.append(f"title: {title}")
        if url:
            meta.append(f"URL: {url}")
        meta_str = " | ".join(meta)
        lines.append(f"[{i}] {meta_str}\n{text}\n")

    chunks_str = "\n".join(lines)

    prompt = f"""
אתה מקבל שאלה של תושב חיפה ורשימת קטעי טקסט (מסומנים כמספרים [1], [2], ...).
עליך לבחור את הקטעים שהכי עוזרים לענות על השאלה.

שאלה:
\"\"\"{question}\"\"\"

קטעים:
{chunks_str}

הוראות:
1. בחר את {top_k} הקטעים הרלוונטיים ביותר.
2. החזר רק רשימה של המספרים, מופרדת בפסיקים, בסדר יורד של רלוונטיות.
3. לדוגמה: 3,1,5,2,4

החזר רק את הרשימה, ללא טקסט נוסף.
"""
    resp = call_gemini(gemini_model, prompt)
    text = resp.strip()
    # try to extract numbers from the response
    import re
    indices = re.findall(r"\d+", text)
    indices = [int(i) for i in indices if int(i) >= 1 and int(i) <= len(chunks)]
    # remove duplicates and keep order
    seen = set()
    ordered = []
    for i in indices:
        if i not in seen:
            seen.add(i)
            ordered.append(i)
    if not ordered:
        # if we didn't manage to parse the response – return the first ones
        return chunks[:top_k]
    # map back to chunks (1-based -> 0-based)
    ordered_chunks = [chunks[i - 1] for i in ordered[:top_k]]
    return ordered_chunks


In [ ]:
# Run RAG with different strategies

def run_rag_strategy(question: str, strategy: str) -> dict:
    """
    strategy options:
    - "orig"
    - "rephrase"
    - "enrich"
    - "rephrase_enrich"
    - "rerank"
    - "optimized" (run combinations and let Gemini choose the best)
    """
    q_orig = question.strip()

    # different query formulations
    q_rephrased = rephrase_query(q_orig) if strategy in ("rephrase", "rephrase_enrich", "optimized") else q_orig
    q_enriched = enrich_query(q_orig) if strategy in ("enrich", "rephrase_enrich", "optimized") else q_orig

    answers = {}

    # --- 1) original query
    def _run_single_rag(query: str, use_rerank: bool = False):
        # retrieval
        chunks = retriever.retrieve(
            query=query,
            top_k=TOP_K_RETRIEVAL,
            exclude_file_types=None,
            include_file_types=None,
        )
        # reranking (optional)
        if use_rerank:
            chunks_final = rerank_chunks(query, chunks, top_k=TOP_K_FINAL)
        else:
            chunks_final = chunks[:TOP_K_FINAL]

        # build the prompt
        prompt = prompt_builder.build_prompt(
            question=query,
            chunks=chunks_final,
            include_sources=True,
        )
        # answer
        answer = call_gemini(gemini_model, prompt)
        return answer.strip(), chunks_final

    # basic strategies
    if strategy == "orig":
        ans, chunks_used = _run_single_rag(q_orig, use_rerank=False)
        return {"answer": ans, "chunks": chunks_used, "used_query": q_orig}

    if strategy == "rephrase":
        ans, chunks_used = _run_single_rag(q_rephrased, use_rerank=False)
        return {"answer": ans, "chunks": chunks_used, "used_query": q_rephrased}

    if strategy == "enrich":
        ans, chunks_used = _run_single_rag(q_enriched, use_rerank=False)
        return {"answer": ans, "chunks": chunks_used, "used_query": q_enriched}

    if strategy == "rephrase_enrich":
        # for example, use the enriched version of the rephrased query, here for simplicity we take the enriched version
        combined = q_rephrased + " " + q_enriched
        ans, chunks_used = _run_single_rag(combined, use_rerank=False)
        return {"answer": ans, "chunks": chunks_used, "used_query": combined}

    if strategy == "rerank":
        ans, chunks_used = _run_single_rag(q_orig, use_rerank=True)
        return {"answer": ans, "chunks": chunks_used, "used_query": q_orig}

    if strategy == "optimized":
        # run some variants
        variants = {
            "orig": _run_single_rag(q_orig, use_rerank=False),
            "rephrase_rerank": _run_single_rag(q_rephrased, use_rerank=True),
            "enrich_rerank": _run_single_rag(q_enriched, use_rerank=True),
        }

        # ask Gemini to choose the best answer
        chooser_prompt = f"""
יש לנו שאלה של משתמש ושלוש תשובות שונות שנוצרו על בסיס מקורות רלוונטיים.

שאלה:
\"\"\"{q_orig}\"\"\"

תשובות:
1) [orig]:
\"\"\"{variants['orig'][0]}\"\"\"

2) [rephrase_rerank]:
\"\"\"{variants['rephrase_rerank'][0]}\"\"\"

3) [enrich_rerank]:
\"\"\"{variants['enrich_rerank'][0]}\"\"\"

בחר את התשובה הטובה ביותר מבחינת דיוק, שלמות ואמינות.
החזר JSON בלבד בפורמט:
{{
  "best": "<orig/rephrase_rerank/enrich_rerank>",
  "final_answer": "<התשובה כפי שאתה ממליץ להציג למשתמש>"
}}
"""
        chooser_resp = call_gemini(gemini_model, chooser_prompt)
        try:
            data = json.loads(chooser_resp)
            best_key = data.get("best", "orig")
            final_answer = data.get("final_answer", variants[best_key][0])
        except Exception:
            best_key = "orig"
            final_answer = variants["orig"][0]

        return {
            "answer": final_answer.strip(),
            "chunks": variants[best_key][1],
            "used_query": q_orig,
            "optimizer_choice": best_key,
        }

    # fallback
    ans, chunks_used = _run_single_rag(q_orig, use_rerank=False)
    return {"answer": ans, "chunks": chunks_used, "used_query": q_orig}


In [ ]:
# Run all strategies on testset

test_df = pd.read_csv(TESTSET_CSV, encoding="utf-8")
print("Questions in testset:", len(test_df))

strategies = ["orig", "rephrase", "enrich", "rephrase_enrich", "rerank", "optimized"]

records = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Questions"):
    q = row["question"]
    gold_answer = row["answer"]
    doc_id = row.get("doc_id", "")
    url = row.get("url", "")
    title = row.get("title", "")

    for strat in strategies:
        out = run_rag_strategy(q, strat)
        answer = out["answer"]
        chunks = out["chunks"]
        used_query = out.get("used_query", q)
        optimizer_choice = out.get("optimizer_choice", "")

        # save the chunk urls for future use
        chunk_urls = [c.get("url", "") for c in chunks]

        records.append({
            "question": q,
            "gold_answer": gold_answer,
            "strategy": strat,
            "rag_answer": answer,
            "used_query": used_query,
            "doc_id": doc_id,
            "url": url,
            "title": title,
            "optimizer_choice": optimizer_choice,
            "chunk_urls": json.dumps(chunk_urls, ensure_ascii=False),
        })

results_df = pd.DataFrame(records)
results_df.head()


In [ ]:
# Save the results

results_df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print("Saved raw RAG results to:", OUT_CSV)
